In [0]:
# Instalacja biblioteki (zostaje bez zmian)
%pip install faker
dbutils.library.restartPython()

import json
import random
import os
from faker import Faker
from datetime import datetime

fake = Faker()

# ZMIANA: Zamiast przestarzałego /dbfs/FileStore, zapisujemy w standardowym katalogu wolumenu /tmp/
BASE_DIR = "/tmp/showcase_raw_data"
HR_DIR = f"{BASE_DIR}/hr_batch"
PAYROLL_DIR = f"{BASE_DIR}/payroll_stream"

# Tworzymy foldery
os.makedirs(HR_DIR, exist_ok=True)
os.makedirs(PAYROLL_DIR, exist_ok=True)

print(f"📁 Foldery utworzone w bezpiecznej ścieżce: {BASE_DIR}")

# ==========================================
# 1. GENERATOR BATCH (HR) - Zagnieżdżony JSON
# ==========================================
hr_data = []
for i in range(1, 6): # Generujemy 5 departamentów
    department = {
        "dept_id": 100 + i,
        "department_name": fake.job(),
        "employees": [
            {
                "emp_id": int(f"{100+i}0{j}"),
                "name": fake.name(),
                "city": fake.city(),
                "updated_at": datetime.now().isoformat()
            } for j in range(1, random.randint(3, 6)) # Zagnieżdżona lista pracowników!
        ]
    }
    hr_data.append(department)

# Zapisujemy jako plik JSON (symulacja zrzutu z bazy HR)
hr_file_path = f"{HR_DIR}/hr_export_{datetime.now().strftime('%Y%m%d')}.json"
with open(hr_file_path, 'w') as f:
    for dept in hr_data:
        f.write(json.dumps(dept) + '\n') # JSON Lines (NDJSON)

print(f"✅ Wygenerowano plik HR Batch: {hr_file_path}")

# ==========================================
# 2. GENERATOR STREAM (PAYROLL) - DATA SKEW
# ==========================================
payroll_data = []
# Generujemy 10,000 logów transakcyjnych
for _ in range(10000):
    # CELOWY DATA SKEW: 80% szans, że transakcja jest z Krakowa (Krakow HQ)
    if random.random() < 0.8:
        office = "Krakow HQ"
    else:
        office = random.choice(["Wroclaw", "Warsaw", "Gdansk", "Poznan"])
        
    log = {
        "transaction_id": fake.uuid4(),
        "emp_id": fake.random_int(min=10101, max=10505),
        "office_location": office,
        "hours_logged": round(random.uniform(4.0, 12.0), 2),
        "timestamp": fake.date_time_this_month().isoformat()
    }
    payroll_data.append(log)

payroll_file_path = f"{PAYROLL_DIR}/payroll_events_{datetime.now().strftime('%Y%m%d%H%M%S')}.json"
with open(payroll_file_path, 'w') as f:
    for log in payroll_data:
        f.write(json.dumps(log) + '\n')

print(f"✅ Wygenerowano zniekształcony (Data Skew) plik Payroll: {payroll_file_path}")
print("🚀 Faza 1 zakończona pełnym sukcesem!")